# Voice of Customer Analysis — Zava

In [ ]:
# Setup
import pandas as pd
import json
import matplotlib.pyplot as plt

print("Voice of Customer Analysis — Zava")

## Corpus Overview

In [ ]:
corpus_stats = {
    'Metric': ['Documents', 'Reviews', 'Support Chats', 'English', 'Spanish', 'French'],
    'Count': [89, 44, 45, 59, 19, 11]
}

stats_df = pd.DataFrame(corpus_stats)
display(stats_df)

plt.figure(figsize=(8, 4))
plt.bar(stats_df['Metric'], stats_df['Count'], color=['#4C78A8', '#F58518', '#54A24B', '#EECA3B', '#B279A2', '#FF9DA7'])
plt.title('Corpus Distribution')
plt.ylabel('Count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Theme Discovery

In [ ]:
theme_data = [
    {
        'Theme': 'Product Quality Complaints',
        'Seed DocId': 2,
        'Top 5 Neighbor DocIds': [7, 18, 21, 3, 8],
        'Precision@5': 0.80
    },
    {
        'Theme': 'Smart Fabric Connectivity Failures',
        'Seed DocId': 44,
        'Top 5 Neighbor DocIds': [39, 43, 40, 42, 41],
        'Precision@5': 1.00
    },
    {
        'Theme': 'Support Complaints & Returns',
        'Seed DocId': 55,
        'Top 5 Neighbor DocIds': [80, 68, 76, 85, 66],
        'Precision@5': 0.80
    },
    {
        'Theme': 'Returns Support Conversations',
        'Seed DocId': 48,
        'Top 5 Neighbor DocIds': [60, 76, 75, 80, 77],
        'Precision@5': 1.00
    }
]

themes_df = pd.DataFrame(theme_data)
display(themes_df)

## Precision Audit Table

In [ ]:
audit_rows = [
    (7, 'Product Quality Complaints', 1, 'Strongly negative review about poor fit, cheap fabric, stitching failure, and a refund request.'),
    (18, 'Product Quality Complaints', 0, 'Positive review with a minor defect note, not a core quality complaint.'),
    (21, 'Product Quality Complaints', 1, 'Explicitly negative about fit, finish, comfort, and overall quality.'),
    (3, 'Product Quality Complaints', 1, 'Complains about poor fit, cheap feel, discomfort, and requests a refund.'),
    (8, 'Product Quality Complaints', 1, 'Negative about fit, premium feel, shape retention, and overall quality.'),
    (39, 'Smart Fabric Connectivity Failures', 1, 'Directly describes smart fabric losing connection after the first wash.'),
    (43, 'Smart Fabric Connectivity Failures', 1, 'Spanish version of the same connectivity failure after washing.'),
    (40, 'Smart Fabric Connectivity Failures', 1, 'Explicitly says the smart features fail after washing and pairing never works.'),
    (42, 'Smart Fabric Connectivity Failures', 1, 'Says the smart features failed after washing and the app no longer detects the garment.'),
    (41, 'Smart Fabric Connectivity Failures', 1, 'Repeatedly loses connection to the app after laundry, matching the theme closely.'),
    (72, 'Smart Fabric Connectivity Failures', 0, 'A product-care question about washing, not a reported connectivity failure.'),
    (84, 'Smart Fabric Connectivity Failures', 1, 'A tech-support exchange about the product no longer connecting to the app.'),
    (80, 'Support Complaints & Returns', 1, 'A return request tied to a smart-fabric product not working as expected.'),
    (68, 'Support Complaints & Returns', 1, 'A complaint support chat about a smart-fabric product failing and needing a replacement.'),
    (62, 'Support Complaints & Returns', 0, 'An order-status inquiry rather than a complaint or return case.')
]

audit_df = pd.DataFrame(audit_rows, columns=['DocId', 'Theme', 'Label', 'Reason'])
display(audit_df)

## False Positive Analysis

- **DocId 18** was a false positive for the Product Quality Complaints theme because it mentioned a small defect and shared product-quality vocabulary, even though it was overall positive and did not reflect a core complaint.
- **DocId 72** was a false positive for Smart Fabric Connectivity Failures because it mentioned washing and product care, which overlapped lexically with the seed theme but did not report a real connectivity issue.
- **DocId 62** was a false positive for Support Complaints & Returns because it appeared in the support-chat neighborhood and referenced a smart-fabric product, but it was an order-status request rather than a complaint or return case.

## Model Card

### 1. System
This system retrieves semantically similar customer and support documents from a multilingual corpus using vector similarity over document embeddings. It is designed to surface related reviews and support chats for themes such as product-quality complaints, smart-fabric connectivity failures, and complaint/return support interactions.

### 2. Corpus
The corpus contains 89 documents drawn from two primary sources: customer reviews and support chat transcripts. The documents span three languages: English, Spanish, and French.

### 3. Precision results
The system was evaluated on three retrieval themes: Product Quality Complaints (precision@5 = 0.80), Smart Fabric Connectivity Failures (precision@5 = 1.00), and Support Complaints & Returns (precision@5 = 0.80).

### 4. Known failure modes
The main failure modes observed in the audit were vocabulary collisions, support-domain boundary errors, and mild semantic drift where a positive or neutral document could still appear near negative ones due to shared product context.

### 5. What this system is good for
It is useful for finding related customer feedback around a theme, surfacing clusters of complaints and support issues, and supporting exploratory analysis across languages.

### 6. What this system should NOT be used for
It should not be used for high-stakes automated triage, exact intent classification, or replacing human review without additional filtering.

### 7. What to test or improve next
Next steps include testing hybrid retrieval, adding metadata-aware filtering, and evaluating reranking methods that better distinguish intent.

In [ ]:
# Architecture Diagram
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 4)
ax.axis('off')

nodes = [
    ('Docs table', 0.8, 2.0),
    ('find_similar_docs_by_doc_id', 3.2, 2.0),
    ('Theme Clusters', 6.0, 2.0),
    ('Precision Audit', 8.6, 2.0),
    ('Model Card', 9.9, 2.0),
]

for label, x, y in nodes:
    ax.text(x, y, label, ha='center', va='center', fontsize=10, bbox=dict(boxstyle='round,pad=0.35', facecolor='#f2f2f2', edgecolor='#4C78A8'))

arrows = [
    (0.8, 2.0, 3.2, 2.0),
    (3.2, 2.0, 6.0, 2.0),
    (6.0, 2.0, 8.6, 2.0),
    (8.6, 2.0, 9.9, 2.0),
]

for x1, y1, x2, y2 in arrows:
    arrow = FancyArrowPatch((x1 + 0.8, y1), (x2 - 0.8, y2), arrowstyle='->', mutation_scale=12, linewidth=1.3, color='#4C78A8')
    ax.add_patch(arrow)

plt.tight_layout()
plt.show()